# Module 6 Homework: Batch Processing (Spark)

## Setup

- **Data**: `yellow_tripdata_2025-11.parquet` (Yellow taxi, November 2025)
- **PySpark version**: 4.1.1
- **Java**: OpenJDK 17 (Temurin-17.0.18+8)
- **JAVA_HOME**: `/home/highview/tools/jdk-17.0.18+8`

> **Note (Spark 4.x)**: Timestamps are `TIMESTAMP_NTZ` type and cannot be cast directly to `BIGINT`.
> Use `F.unix_timestamp(col.cast('timestamp'))` to convert to seconds for duration calculations.

---

## Q1: Install Spark and PySpark

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('homework6') \
    .getOrCreate()

print(spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/14 15:15:27 WARN Utils: Your hostname, Ivys-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.193 instead (on interface en0)
26/04/14 15:15:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 15:15:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.1.1


**Answer: `4.1.1`**

---

## Q2: Yellow November 2025 — Average Parquet File Size
Read the November 2025 Yellow into a Spark Dataframe.

Repartition the Dataframe to 4 partitions and save it to parquet.

What is the average size of the Parquet (ending with .parquet extension) Files that were created (in MB)? Select the answer which most closely matches.

- 6MB
- 25MB
- 75MB
- 100MB

In [4]:
df = spark.read.parquet('2025_yellow_data/yellow_tripdata_2025-11.parquet')
print(f"Total rows: {df.count()}")  # 4,181,444

df_repartitioned = df.repartition(4)
df_repartitioned.write.mode('overwrite').parquet('output/')

!ls -lh output/*.parquet

Total rows: 4181444


-rw-r--r--  1 ivylu  staff    24M Apr 14 15:18 output/part-00000-bb991f77-d7c6-4a31-99ca-b196a954066a-c000.snappy.parquet
-rw-r--r--  1 ivylu  staff    24M Apr 14 15:18 output/part-00001-bb991f77-d7c6-4a31-99ca-b196a954066a-c000.snappy.parquet
-rw-r--r--  1 ivylu  staff    24M Apr 14 15:18 output/part-00002-bb991f77-d7c6-4a31-99ca-b196a954066a-c000.snappy.parquet
-rw-r--r--  1 ivylu  staff    24M Apr 14 15:18 output/part-00003-bb991f77-d7c6-4a31-99ca-b196a954066a-c000.snappy.parquet


Answer: 25MB

---

## Question 3: Count records

How many taxi trips were there on the 15th of November?

Consider only trips that started on the 15th of November.

- 62,610
- 102,340
- 162,604
- 225,768


In [5]:
# check the column names
df.printSchema()
# print the first 5 rows truncate=False is to avoid the truncation of the column names
df.show(5, truncate=False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [13]:
from pyspark.sql.functions import to_date

nov_15_count = df.filter(to_date("tpep_pickup_datetime") == "2025-11-15").count()

print(f"Q3 ANSWER: {nov_15_count:,} trips on November 15, 2025")



26/03/18 17:28:56 WARN Executor: Issue communicating with driver in heartbeater]
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:81)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:674)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1363)
	at org.apache.spark.executor.Executor.$anonfun$heartbeater$1(Executor.scala:356)
	at scala.runtime.java8.JFunction0$mcV$sp.apply(JFunction0$mcV$sp.scala:18)
	at org.apache.spark.util.Utils$.logUncaughtExceptions(Utils.scala:194

Q3 ANSWER: 162,604 trips on November 15, 2025
Closest option: 162,604


26/03/18 17:29:06 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

---

## Q4: Longest Trip in Hours

# Note: TIMESTAMP_NTZ requires casting to timestamp before unix_timestamp()
result = df.withColumn(
    'duration_hours',
    (F.unix_timestamp(df.tpep_dropoff_datetime.cast('timestamp'))
     - F.unix_timestamp(df.tpep_pickup_datetime.cast('timestamp'))) / 3600
).agg(F.max('duration_hours').alias('max_hours'))

#result:90.65hrs
```

**Answer: 90.6**

---

## Q5: Spark UI Port

Spark's User Interface dashboard runs on local port:

**Answer: `4040`**

---

## Q6: Least Frequent Pickup Location Zone

```python
zones = spark.read.option("header", "true").csv('taxi_zone_lookup.csv')
df.createOrReplaceTempView('yellow_2025_11')
zones.createOrReplaceTempView('zones')

spark.sql("""
SELECT z.Zone, COUNT(1) as trip_count
FROM yellow_2025_11 y
LEFT JOIN zones z ON y.PULocationID = CAST(z.LocationID AS INT)
GROUP BY z.Zone
ORDER BY trip_count ASC
LIMIT 5
""").show(truncate=False)
```

Results:
| Zone | trip_count |
|------|------------|
| Governor's Island/Ellis Island/Liberty Island | 1 |
| Eltingville/Annadale/Prince's Bay | 1 |
| Arden Heights | 1 |
| Port Richmond | 3 |
| Rikers Island | 4 |


**Answer: `Governor's Island/Ellis Island/Liberty Island`**